# Quickstart

This quickstart guide helps you add **foliplus** plugins to a Folium map in just a few minutes.

In this notebook, you will:
- Install `foliplus`
- Build runnable sample geospatial data
- Enable common interactive plugins (search, layer toggle, heatmap, scale, measure, fullscreen)
- Render the map in Notebook and optionally export it as HTML

Run cells from top to bottom.

## Installation

```bash
pip install -U foliplus
```

If you are using a fresh environment, restart the notebook kernel after installation before continuing.

## Dependencies

- `folium >= 0.14`
- `my-data-toolkit` (optional, used in this guide for `from_xy` and `explore` convenience)

Core dependencies are installed automatically with `foliplus`. You can install the optional ones via:

```bash
pip install -U my-data-toolkit
```

## Data Preparation & Base Map

We will create a comprehensive scenario to better showcase **foliplus** features:
1. **Multi-source Data**: Mix of district points and simulated facility points.
2. **Layer Hierarchy**: Organize data into different categories to demonstrate `LayerControl`.
3. **Property Aggregation**: Include numeric fields (e.g., `value`) to demonstrate `HeatmapControl`'s weighted aggregation.
4. **Rich Overlays**: Use various markers and styles to show `LayerControl`'s geometry-type icons.

In [ ]:
import dtoolkit.geoaccessor  # noqa: F401
import folium
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Build Sample Data
# ---------------------------------------------------------------------------
# Shanghai district centers
districts = pd.DataFrame(
    {
        "name": ["Huangpu", "Xuhui", "Pudong", "Minhang", "Jing'an"],
        "population": [660000, 1110000, 5680000, 2650000, 980000],
        "x": [121.4737, 121.4365, 121.5440, 121.3817, 121.4487],
        "y": [31.2304, 31.1886, 31.2213, 31.1128, 31.2288],
    }
).from_xy("x", "y", crs=4326)

# Simulated facility points (e.g., shops or charging stations)
np.random.seed(42)
facilities = pd.DataFrame(
    {
        "id": range(200),
        "value": np.random.randint(10, 100, 200),
        "x": np.random.uniform(121.2, 121.7, 200),
        "y": np.random.uniform(31.0, 31.4, 200),
    }
).from_xy("x", "y", crs=4326)

# ---------------------------------------------------------------------------
# 2. Initialize Map
# ---------------------------------------------------------------------------
center = districts.geocentroid()
m = folium.Map(location=[center.y, center.x], zoom_start=10, tiles=None)

# Add multiple base layers to test LayerControl's base layer switching
folium.TileLayer("CartoDB positron", name="Light Theme").add_to(m)
folium.TileLayer("CartoDB dark_matter", name="Dark Theme").add_to(m)
folium.TileLayer("OpenStreetMap", name="Standard Map").add_to(m)

# ---------------------------------------------------------------------------
# 3. Add Thematic Layers
# ---------------------------------------------------------------------------
# Layer 1: Districts (Markers)
districts.explore(
    m=m,
    name="District Centers",
    column="name",
    marker_type="marker",
    popup=True,
    tooltip="name",
    legend=False,
)

# Layer 2: Facilities (Circles) - This will be used for Heatmap aggregation later
facilities.explore(
    m=m,
    name="Facility Points",
    column="value",
    cmap="Reds",
    marker_type="circle_marker",
    style_kwds={"radius": 5, "fillOpacity": 0.8},
    legend=False,
)

print("✅ Map and Layers are ready")

## Enable foliplus Plugins

Now we add the controls. Notice how they interact with the data we just prepared:

- **`LayerControl`**: You can reorder the "Facility Points" and "District Centers", or pick a custom background color.
- **`HeatmapControl`**: Try switching the `Aggregation` to `sum` and `Field` to `value` to see a weighted heatmap of facilities.
- **`MapSearch`**: Search for "Pudong" or specific coordinates like "121.5,31.2" to fly to.
- **`MeasureControl`**: Measure distances between facilities or create buffer zones.
- **`Fullscreen`**: Experience the map in its full glory.

In [ ]:
from foliplus import (
    Fullscreen,
    HeatmapControl,
    LayerControl,
    MapSearch,
    MeasureControl,
    ScaleControl,
)

# Add all plugins with default settings to keep it simple but powerful
MapSearch(locale="en").add_to(m)
LayerControl(locale="en").add_to(m)
HeatmapControl(locale="en").add_to(m)
ScaleControl(locale="en").add_to(m)
MeasureControl(locale="en").add_to(m)
Fullscreen(locale="en").add_to(m)

print("✅ All foliplus plugins are active")

## Display Map

Run the next cell to render the interactive map in Notebook.

If you want to share the result, use the optional HTML export section below.

In [ ]:
m

## Export to HTML

If you want to view the map in a browser or host it as a static file, export it to HTML.

In [ ]:
file = "map.html"
m.save(file)
print(f"✅ Saved: {file}")

## Common Issues

- **Map not showing**: Make sure the map display cell (Cell 9) has been executed and the notebook is trusted in your editor.
- **`ModuleNotFoundError: dtoolkit`**: Run `pip install my-data-toolkit` and restart the kernel. This is an optional dependency used in this guide for easier data handling.
- **Plugin buttons missing**: Ensure the "Enable foliplus Plugins" cell (Cell 7) ran successfully without errors.

## Next Steps

- **API Documentation**: Check the [API Reference](api.rst) for detailed parameters of each plugin.
- **Customization**: Experiment with `HeatmapControl(color_scheme="Viridis")` or `LayerControl(position="topright")`.
- **Real Data**: Replace the simulated datasets with your own GeoPandas DataFrames and start exploring!